In [1]:
import pandas as pd

markdown_string = """
| TokenMixer    | Kernel Size | JSRT           | Graz            | Tiger@768²                         |
|:--------------|-------------|----------------|-----------------|------------------------------------|
| pooling       | 3           | 0.9468, 7.8257 | 0.8094, 9.3323  | 0.6125, 482.6353                   |
|               | 5           | 0.9463, 8.2299 | 0.8072, 9.4621  | 0.5555, 495.7311                   |
|               | 7           | 0.945, 8.192   | 0.7973, 9.5939  | 0.558, 534.8672                    |
|               | 9           | -              | -               | 0.6019, 445.0148                   |
| conv          | 3           | 0.9486, 8.2723 | 0.8164, 9.6074  | 0.6024, 544.4164                   |
|               | 5           | 0.951, 8.8047  | 0.8276, 8.6283  | 0.5622, 503.8582                   |
|               | 7           | 0.9504, 8.5596 | 0.833, 7.2717   | 0.5623, 531.7621                   |
|               | 9           | -              | -               | 0.5689, 514.4671                   |
| sep_conv      | 3           | 0.95, 8.1622   | 0.8119, 9.238   | 0.5791, 507.2356                   |
|               | 5           | 0.9495, 8.5882 | 0.8319, 8.1215  | 0.5941, 487.0254                   |
|               | 7           | 0.9504, 8.5015 | 0.832, 7.6763   | 0.5711, 554.1542                   |
|               | 9           | -              | -               | 0.5893, 499.8528                   |
| locAttn       | 3           | 0.9418, 7.6488 | 0.8165, 10.3977 | 0.5167, 627.1148                   |
|               | 5           | 0.9465, 8.094  | 0.7957, 9.055   | 0.5303, 547.0928                   |
|               | 7           | 0.9449, 7.1625 | 0.7927, 9.1572  | 0.5374, 584.6804                   |
|               | 9           | -              | -               | 0.5511, 515.2096                   |
| fullAttn      | -           | 0.9443, 7.5997 | 0.7562, 20.5518 | - |
| identity      | 1           | 0.9458, 7.7736 | 0.7417, 19.0719 | 0.5358, 535.9729                   |
| UNet          | 3           | 0.9552, 5.1958 | 0.8478, 13.1739 | 0.5666, 565.8803                   |
|               | 5           | 0.9499, 6.9125 | 0.8258, 16.4765 | 0.5458, 551.4406                   |
|               | 7           | 0.9505, 5.5495 | 0.8318, 17.5944 | 0.5533, 553.0115                   |
|               | 9           | -              | -               | 0.5544, 551.8779                   |
| UNet@PatchEmb | 3           | 0.9357, 8.291  | 0.7738, 10.5785 | 0.5735, 589.6542                   |
|               | 5           | 0.9342, 7.1374 | 0.7864, 8.7964  | 0.5981, 540.7787                   |
|               | 7           | 0.9335, 8.3351 | 0.7629, 9.6254  | 0.6047, 499.9212                   |
|               | 9           | -              | -               | 0.5639, 549.7401                   |
"""

lines = markdown_string.split("\n")
header = lines[1].strip("|").split("|")
header = list(map(lambda s: s.strip(), header))

data = []

# Loop through lines starting from 2
for line in lines[3:]:

    # Break once we hit an empty line
    if not line.strip():
        break

    cols = line.strip("|").split("|")
    cols = map(lambda s: s.strip(), cols)
    row = dict(zip(header, cols))
    data.append(row)

df = pd.DataFrame(data)
df.iloc[:, 0] = df.iloc[:, 0].replace("", pd.NA).ffill()
df.set_index(['TokenMixer', 'Kernel Size'], inplace=True)

df_split = df.apply(lambda col: col.str.split(","))
df_expanded = pd.concat(
    [df_split[col].apply(pd.Series).add_prefix(f"{col.strip()}_") for col in df_split.columns],
    axis=1
)

suffix_map = {
    "_0": "_dsc",
    "_1": "_hdd95"
}
df = df_expanded.rename(columns=lambda col: next((col.replace(k, v) for k, v in suffix_map.items() if k in col), col))
df = df.apply(pd.to_numeric, errors="coerce")

# drop unet and other metrics
df_rank = df.loc[~df.index.get_level_values(0).str.startswith('UNet'), df.columns.str.endswith('_dsc')]
df_rank = df_rank.rank(axis=0, ascending=False, method="first")
df_rank_no_imgwoof = df_rank.iloc[:, 1:]

df_rank


JSRT_dsc  Graz_dsc  Tiger@768²_dsc
TokenMixer Kernel Size                                    
pooling    3                 7.0       8.0             1.0
           5                 9.0       9.0            12.0
           7                11.0      10.0            11.0
           9                 NaN       NaN             3.0
conv       3                 6.0       6.0             2.0
           5                 1.0       4.0            10.0
           7                 2.0       1.0             9.0
           9                 NaN       NaN             8.0
sep_conv   3                 4.0       7.0             6.0
           5                 5.0       3.0             4.0
           7                 3.0       2.0             7.0
           9                 NaN       NaN             5.0
locAttn    3                14.0       5.0            17.0
           5                 8.0      11.0            16.0
           7                12.0      12.0            14.0
           9                 NaN       NaN            13.0
fullAttn   -                13.0      13.0             NaN
identity   1                10.0      14.0            15.0

# Pool Size ranking

In [2]:
df_rank.groupby('Kernel Size').mean()

,JSRT_dsc,Graz_dsc,Tiger@768²_dsc
Kernel Size,,,
-,13.00,13.00,NaN
1,10.00,14.00,15.00
3,7.75,6.50,6.50
5,5.75,6.75,10.50
7,7.00,6.25,10.25
9,NaN,NaN,7.25


# Token Mixer

In [3]:
df_rank.groupby('TokenMixer').mean()

,JSRT_dsc,Graz_dsc,Tiger@768²_dsc
TokenMixer,,,
conv,3.000000,3.666667,7.25
fullAttn,13.000000,13.000000,NaN
identity,10.000000,14.000000,15.00
locAttn,11.333333,9.333333,15.00
pooling,9.000000,9.000000,6.75
sep_conv,4.000000,4.000000,5.50


In [4]:
df_rank.groupby('TokenMixer').mean().mean(1).sort_values()

TokenMixer
sep_conv     4.500000
conv         4.638889
pooling      8.250000
locAttn     11.888889
identity    13.000000
fullAttn    13.000000
dtype: float64

In [8]:
df_print = df.loc[:, df.columns.str.endswith('_dsc')]
df_print = df_print.rename(columns=lambda s: s[:-4])
df_print = df_print.style.format(precision=4, na_rep="")
print(df_print.to_latex())

\begin{tabular}{llrrr}
 &  & JSRT & Graz & Tiger@768² \\
TokenMixer & Kernel Size &  &  &  \\
\multirow[c]{4}{*}{pooling} & 3 & 0.9468 & 0.8094 & 0.6125 \\
 & 5 & 0.9463 & 0.8072 & 0.5555 \\
 & 7 & 0.9450 & 0.7973 & 0.5580 \\
 & 9 &  &  & 0.6019 \\
\multirow[c]{4}{*}{conv} & 3 & 0.9486 & 0.8164 & 0.6024 \\
 & 5 & 0.9510 & 0.8276 & 0.5622 \\
 & 7 & 0.9504 & 0.8330 & 0.5623 \\
 & 9 &  &  & 0.5689 \\
\multirow[c]{4}{*}{sep_conv} & 3 & 0.9500 & 0.8119 & 0.5791 \\
 & 5 & 0.9495 & 0.8319 & 0.5941 \\
 & 7 & 0.9504 & 0.8320 & 0.5711 \\
 & 9 &  &  & 0.5893 \\
\multirow[c]{4}{*}{locAttn} & 3 & 0.9418 & 0.8165 & 0.5167 \\
 & 5 & 0.9465 & 0.7957 & 0.5303 \\
 & 7 & 0.9449 & 0.7927 & 0.5374 \\
 & 9 &  &  & 0.5511 \\
fullAttn & - & 0.9443 & 0.7562 &  \\
identity & 1 & 0.9458 & 0.7417 & 0.5358 \\
\multirow[c]{4}{*}{UNet} & 3 & 0.9552 & 0.8478 & 0.5666 \\
 & 5 & 0.9499 & 0.8258 & 0.5458 \\
 & 7 & 0.9505 & 0.8318 & 0.5533 \\
 & 9 &  &  & 0.5544 \\
\multirow[c]{4}{*}{UNet@PatchEmb} & 3 & 0.9357 & 0.7738 